# DNB — Offline Smoke Tests

Interactive validation of the pipeline:

```
WaveletConvolution → TargetWaveDetector (activation) → AmplitudeMonitor (inhibition) → StimTrigger
```

All parameters are defined once in the config dict below.

**Event semantics:**
- `SLOW_WAVE` — detection event (logged, never a stim)
- `STIM` — audio stimulation at a predicted future peak (`pulse_index` 1-indexed)

In [ ]:
from math import pi

# ── All pipeline parameters in one place ──────────────────────────────
#
# Mirrors the YAML config structure so the notebook and config.yaml
# stay in sync. Every test cell reads from CFG rather than defining
# its own magic numbers.

CFG = {
    'pipeline': {
        'sample_rate': 1000.0,
        'n_channels': 1,
        'buffer_duration': 10.0,
        'chunk_duration': 0.05,   # 50 ms — see notes in README on latency
    },

    'wavelet': {
        'freq_min': 0.5,
        'freq_max': 30.0,
        'n_freqs': 10,
        'n_cycles_base': 3.0,
    },

    'target_wave': {
        'id': 'slow_wave',
        'freq_range': (0.5, 2.0),
        'target_phase': 0.0,       # 0 = positive peak
        'phase_tolerance': 0.15,
        'amp_min': 50.0,
        'amp_max': 10000.0,
        'warmup_chunks': 3,
        'amp_smoothing': 5,
    },

    'amplitude_monitor': {
        'enabled': True,
        'id': 'ied_monitor',
        'freq_range': (80.0, 120.0),   # broadband IED check
        'threshold': None,              # None = adaptive
        'adaptive_n_std': 3.0,
        'warmup_chunks': 20,
        'baseline_chunks': 100,
    },

    'trigger': {
        'activation_detector_id': 'slow_wave',
        'inhibition_detector_id': 'ied_monitor',
        'n_pulses': 1,
        'backoff_s': 5.0,
        'inhibition_cooldown_s': 5.0,
    },

    'synthetic': {
        'duration_s': 120.0,
        'snr': 5.0,
        'n_slow_waves': 15,
        'n_ieds': 40,
        'sw_amplitude': 500.0,
        'ied_amplitude': 3000.0,
        'seed': 42,
    },

    'validation': {
        'time_tolerance': 0.5,
        'snr_levels': [1.0, 2.0, 3.0, 5.0, 10.0],
    },
}

print('Config loaded.')
print(f"  chunk_duration = {CFG['pipeline']['chunk_duration']}s")
print(f"  target_phase   = {CFG['target_wave']['target_phase']:.2f} rad")
print(f"  n_pulses       = {CFG['trigger']['n_pulses']}")
print(f"  IED band       = {CFG['amplitude_monitor']['freq_range']}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from dnb import Pipeline, FileSource, PipelineConfig, EventType
from dnb.modules import (
    WaveletConvolution, TargetWaveDetector, AmplitudeMonitor, StimTrigger,
)
from dnb.validation.ground_truth import validate, Annotation
from test_data import TestData

import dnb
print(f'DNB v{dnb.__version__}')

td = TestData()  # temp dir, auto-cleaned
# td = TestData('test_output')  # persistent dir if you want to inspect files

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['figure.dpi'] = 100

In [ ]:
# ── Helpers: build pipeline components from CFG ───────────────────────

def make_pipeline_config():
    p = CFG['pipeline']
    return PipelineConfig(
        sample_rate=p['sample_rate'],
        n_channels=p['n_channels'],
        buffer_duration=p['buffer_duration'],
        chunk_duration=p['chunk_duration'],
    )

def make_wavelet():
    w = CFG['wavelet']
    return WaveletConvolution(
        freq_min=w['freq_min'], freq_max=w['freq_max'],
        n_freqs=w['n_freqs'], n_cycles_base=w['n_cycles_base'],
    )

def make_detector(**overrides):
    tw = {**CFG['target_wave'], **overrides}
    return TargetWaveDetector(
        id=tw['id'], freq_range=tw['freq_range'],
        target_phase=tw['target_phase'],
        phase_tolerance=tw['phase_tolerance'],
        amp_min=tw['amp_min'], amp_max=tw['amp_max'],
        warmup_chunks=tw['warmup_chunks'],
        amp_smoothing=tw['amp_smoothing'],
    )

def make_inhibitor(**overrides):
    am = {**CFG['amplitude_monitor'], **overrides}
    return AmplitudeMonitor(
        id=am['id'], freq_range=am['freq_range'],
        threshold=am['threshold'],
        adaptive_n_std=am['adaptive_n_std'],
        warmup_chunks=am['warmup_chunks'],
        baseline_chunks=am['baseline_chunks'],
    )

def make_trigger(**overrides):
    tr = {**CFG['trigger'], **overrides}
    return StimTrigger(
        activation_detector_id=tr['activation_detector_id'],
        inhibition_detector_id=tr['inhibition_detector_id'],
        n_pulses=tr['n_pulses'],
        backoff_s=tr['backoff_s'],
        inhibition_cooldown_s=tr['inhibition_cooldown_s'],
    )

def get_detections(events):
    """Get SLOW_WAVE detection events."""
    return [e for e in events if e.event_type == EventType.SLOW_WAVE]

def get_stims(events, pulse_index=None):
    """Get STIM events, optionally filtered by pulse_index (1-indexed)."""
    out = [e for e in events if e.event_type == EventType.STIM]
    if pulse_index is not None:
        out = [e for e in out if e.metadata.get('pulse_index') == pulse_index]
    return out

print('Helpers ready.')

---
## 1. Clean sine wave — verify phase detection

With n_pulses=1 on a clean sine, each detection should produce a SLOW_WAVE
at the target phase, plus a STIM one period later.

In [ ]:
path = td.clean_sine()

pipeline = Pipeline(
    source=FileSource(str(path)),
    modules=[
        make_wavelet(),
        make_detector(amp_min=100.0, amp_max=100000.0, phase_tolerance=0.1),
        make_trigger(n_pulses=1, inhibition_detector_id=None, backoff_s=0.0),
    ],
    config=make_pipeline_config(),
)

events = pipeline.run_offline()
detections = get_detections(events)
stims = get_stims(events)

print(f'{len(detections)} SLOW_WAVE detections, {len(stims)} STIM events')
if detections:
    phases = np.array([e.metadata['phase'] for e in detections])
    det_times = np.array([e.timestamp for e in detections])
    stim_times = np.array([e.timestamp for e in stims])
    target = CFG['target_wave']['target_phase']
    print(f'Detection phase: mean={np.mean(phases):.3f} std={np.std(phases):.3f} (target={target:.3f})')

    # Check that stims land one period after detections
    if stims:
        for d, s in zip(detections[:5], stims[:5]):
            freq = d.metadata['frequency']
            expected_delay = 1.0 / freq
            actual_delay = s.timestamp - d.timestamp
            print(f'  det={d.timestamp:.3f}s stim={s.timestamp:.3f}s '
                  f'delay={actual_delay:.3f}s (expected={expected_delay:.3f}s)')

    data = np.load(str(path))
    sig = data['continuous'][0]
    t = np.arange(len(sig)) / CFG['pipeline']['sample_rate']

    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    axes[0].plot(t, sig, 'b-', lw=0.5, alpha=0.7)
    for e in detections:
        axes[0].axvline(e.timestamp, color='green', alpha=0.4, lw=1)
    for e in stims:
        axes[0].axvline(e.timestamp, color='red', alpha=0.6, lw=1, ls='--')
    axes[0].set_ylabel('Amplitude (µV)')
    axes[0].set_title('Clean 1 Hz sine — green=detection, red=stim')

    axes[1].scatter(det_times, phases, c='green', s=20, zorder=5, label='detection phase')
    axes[1].axhline(target, color='green', ls='--', label=f'target={target:.2f}')
    axes[1].set_ylabel('Phase (rad)')
    axes[1].set_xlabel('Time (s)')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

---
## 2. Synthetic slow waves — detection + validation

In [ ]:
syn = CFG['synthetic']
path_sw, gt_events = td.slow_waves(snr=syn['snr'])
sw_gt = [e for e in gt_events if e.metadata.get('type') == 'SW']
print(f'Planted: {len(sw_gt)} slow waves')

data = np.load(str(path_sw))
sig = data['continuous'][0]
t = np.arange(len(sig)) / CFG['pipeline']['sample_rate']

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(t, sig, 'b-', lw=0.3, alpha=0.6)
for e in sw_gt:
    ax.axvline(e.timestamp, color='green', alpha=0.5, lw=1.5,
               label='planted SW' if e == sw_gt[0] else '')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title(f'Synthetic recording (SNR={syn["snr"]})')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
pipeline = Pipeline(
    source=FileSource(str(path_sw)),
    modules=[
        make_wavelet(),
        make_detector(),
        make_trigger(n_pulses=1, inhibition_detector_id=None, backoff_s=2.0),
    ],
    config=make_pipeline_config(),
)

all_events = pipeline.run_offline()
detections = get_detections(all_events)
stims = get_stims(all_events)
print(f'Detections: {len(detections)}, Stims: {len(stims)}')

# Validate detections against ground truth
annotations = [Annotation(timestamp=e.timestamp, channel=e.channel_id, event_type='SW') for e in sw_gt]
report = validate(detections, annotations, time_tolerance=CFG['validation']['time_tolerance'])
print(report.summary())

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(t, sig, 'b-', lw=0.3, alpha=0.5)
for e in sw_gt:
    ax.axvline(e.timestamp, color='green', alpha=0.4, lw=2,
               label='GT' if e == sw_gt[0] else '')
for i, e in enumerate(detections):
    ax.axvline(e.timestamp, color='orange', alpha=0.6, lw=1, ls='-',
               label='detection' if i == 0 else '')
for i, e in enumerate(stims):
    ax.axvline(e.timestamp, color='red', alpha=0.6, lw=1, ls='--',
               label='stim' if i == 0 else '')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title('Detections (orange) + Stims (red dashed) vs Ground Truth (green)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. N-pulse stimulation

- `n_pulses=0` → SLOW_WAVE detections only, no STIMs
- `n_pulses=1` → 1 STIM at next predicted peak after each detection
- `n_pulses=3` → 3 STIMs at next 3 predicted peaks

In [ ]:
for n_pulses in [0, 1, 3]:
    pipe = Pipeline(
        source=FileSource(str(path_sw)),
        modules=[
            make_wavelet(),
            make_detector(),
            make_trigger(n_pulses=n_pulses, inhibition_detector_id=None, backoff_s=5.0),
        ],
        config=make_pipeline_config(),
    )
    dets = pipe.run_offline()
    sw_dets = get_detections(dets)
    all_stims = get_stims(dets)
    print(f'n_pulses={n_pulses}: {len(sw_dets)} detections, {len(all_stims)} stims')

    if n_pulses > 0 and all_stims:
        # Show first few detection→stim sequences
        for d in sw_dets[:3]:
            det_t = d.timestamp
            freq = d.metadata['frequency']
            my_stims = [s for s in all_stims
                        if abs(s.metadata.get('detection_time', -1) - det_t) < 0.001]
            print(f'  det t={det_t:.3f}s freq={freq:.2f}Hz → {len(my_stims)} stim(s):')
            for s in my_stims:
                print(f'    stim {s.metadata["pulse_index"]}/{n_pulses} '
                      f't={s.timestamp:.3f}s (Δ={s.timestamp - det_t:.3f}s)')
    print()

In [ ]:
# Visual: 3-pulse stim overlaid on the signal

pipe_3 = Pipeline(
    source=FileSource(str(path_sw)),
    modules=[
        make_wavelet(),
        make_detector(),
        make_trigger(n_pulses=3, inhibition_detector_id=None, backoff_s=5.0),
    ],
    config=make_pipeline_config(),
)
dets_3 = pipe_3.run_offline()
detections_3 = get_detections(dets_3)
stims_3 = get_stims(dets_3)

pulse_colors = {1: 'red', 2: 'orange', 3: 'purple'}
labels_used = set()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(t, sig, 'b-', lw=0.3, alpha=0.5)
for e in sw_gt:
    ax.axvline(e.timestamp, color='green', alpha=0.3, lw=2,
               label='GT' if e == sw_gt[0] else '')
for i, e in enumerate(detections_3):
    ax.axvline(e.timestamp, color='gray', alpha=0.5, lw=1,
               label='detection' if i == 0 else '')
for e in stims_3:
    pi_idx = e.metadata['pulse_index']
    c = pulse_colors.get(pi_idx, 'gray')
    lbl = f'stim {pi_idx}' if pi_idx not in labels_used else ''
    labels_used.add(pi_idx)
    ax.axvline(e.timestamp, color=c, alpha=0.7, lw=1.2, ls='--', label=lbl)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title('3-pulse stimulation — detection (gray) + stims at predicted peaks')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

### 3b. Verify scheduled pulse timing

Stim k should land at `detection_time + k/freq`. Check the spacing.

In [ ]:
from collections import defaultdict

# Group stims by their detection_time
sequences = defaultdict(list)
for e in stims_3:
    key = e.metadata['detection_time']
    sequences[key].append(e)

print(f'{len(sequences)} detection sequences with 3-pulse stim:\n')
for det_t, seq in sorted(sequences.items()):
    seq.sort(key=lambda e: e.metadata['pulse_index'])
    freq = seq[0].metadata['frequency']
    period = 1.0 / freq

    print(f'  det_t={det_t:.3f}s  freq={freq:.2f}Hz  period={period:.3f}s')
    for e in seq:
        k = e.metadata['pulse_index']
        expected_t = det_t + k * period
        error_ms = (e.timestamp - expected_t) * 1000
        print(f'    stim {k}: t={e.timestamp:.3f}s  expected={expected_t:.3f}s  error={error_ms:+.1f}ms')

# Aggregate timing accuracy
all_errors = []
for det_t, seq in sequences.items():
    freq = seq[0].metadata['frequency']
    period = 1.0 / freq
    for e in seq:
        k = e.metadata['pulse_index']
        expected_t = det_t + k * period
        all_errors.append(e.timestamp - expected_t)

if all_errors:
    errs = np.array(all_errors) * 1000  # ms
    print(f'\nStim timing error: {np.mean(errs):.1f} ± {np.std(errs):.1f} ms')
    print(f'  (quantised to chunk_duration={CFG["pipeline"]["chunk_duration"]*1000:.0f} ms)')

---
## 4. IED inhibition

The AmplitudeMonitor bandpasses 80–120 Hz and checks RMS power.
When it exceeds the adaptive threshold, stimulation is blocked.

In [ ]:
path_ied, gt_ied = td.slow_waves_with_ieds(
    snr=syn['snr'],
    ied_amplitude=syn['ied_amplitude'],
)
sw_gt_ied = [e for e in gt_ied if e.metadata.get('type') == 'SW']
ied_gt = [e for e in gt_ied if e.metadata.get('type') == 'IED']
print(f'Planted: {len(sw_gt_ied)} SWs, {len(ied_gt)} IEDs')

data_ied = np.load(str(path_ied))
sig_ied = data_ied['continuous'][0]
t_ied = np.arange(len(sig_ied)) / CFG['pipeline']['sample_rate']

# WITH inhibition
pipe_inh = Pipeline(
    source=FileSource(str(path_ied)),
    modules=[
        make_wavelet(),
        make_detector(),
        make_inhibitor(),
        make_trigger(n_pulses=1, backoff_s=5.0),
    ],
    config=make_pipeline_config(),
)
events_inh = pipe_inh.run_offline()
stims_inh = get_stims(events_inh)

# WITHOUT inhibition
pipe_no = Pipeline(
    source=FileSource(str(path_ied)),
    modules=[
        make_wavelet(),
        make_detector(),
        make_trigger(n_pulses=1, inhibition_detector_id=None, backoff_s=3.0),
    ],
    config=make_pipeline_config(),
)
events_no = pipe_no.run_offline()
stims_no = get_stims(events_no)

def near_ieds(stims, ieds, win=1.0):
    return sum(1 for s in stims if any(abs(s.timestamp - i.timestamp) < win for i in ieds))

print(f'With inhibition:    {len(stims_inh)} stims, {near_ieds(stims_inh, ied_gt)} near IEDs')
print(f'Without inhibition: {len(stims_no)} stims, {near_ieds(stims_no, ied_gt)} near IEDs')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for ax, stims, label in [
    (axes[0], stims_no, 'WITHOUT inhibition'),
    (axes[1], stims_inh, 'WITH inhibition'),
]:
    ax.plot(t_ied, sig_ied, 'b-', lw=0.3, alpha=0.5)
    for e in sw_gt_ied:
        ax.axvline(e.timestamp, color='green', alpha=0.3, lw=2)
    for e in ied_gt:
        ax.axvspan(e.timestamp - 0.1, e.timestamp + 0.1, color='orange', alpha=0.3)
    for e in stims:
        ax.axvline(e.timestamp, color='red', alpha=0.7, lw=1, ls='--')
    ax.set_ylabel('Amp')
    ax.set_title(label)
axes[1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

### 4b. N-pulse with IED inhibition

Verify that inhibition cancels pending scheduled stims.

In [ ]:
pipe_3_inh = Pipeline(
    source=FileSource(str(path_ied)),
    modules=[
        make_wavelet(),
        make_detector(),
        make_inhibitor(),
        make_trigger(n_pulses=3, backoff_s=5.0),
    ],
    config=make_pipeline_config(),
)
dets_3_inh = pipe_3_inh.run_offline()
sw_dets_3_inh = get_detections(dets_3_inh)
stims_3_inh = get_stims(dets_3_inh)

# Count complete vs incomplete sequences
seqs_inh = defaultdict(list)
for e in stims_3_inh:
    seqs_inh[e.metadata['detection_time']].append(e)

# A detection might have 0 stims (all cancelled), <3, or 3
complete = sum(1 for s in seqs_inh.values() if len(s) == 3)
incomplete = sum(1 for s in seqs_inh.values() if 0 < len(s) < 3)
no_stims = len(sw_dets_3_inh) - len(seqs_inh)

print(f'3-pulse with IED inhibition:')
print(f'  {len(sw_dets_3_inh)} detections')
print(f'  {complete} complete (all 3 stims delivered)')
print(f'  {incomplete} partial (some stims cancelled)')
print(f'  {no_stims} fully cancelled (0 stims delivered)')
print(f'  {len(stims_3_inh)} total stim events')

---
## 5. SNR sweep

In [ ]:
sweep = td.snr_sweep(snr_levels=CFG['validation']['snr_levels'])
results = []

for actual_snr, path, gt in sweep:
    pipe = Pipeline(
        source=FileSource(str(path)),
        modules=[
            make_wavelet(),
            make_detector(target_phase=pi, phase_tolerance=0.05, amp_min=100.0),
            make_trigger(n_pulses=0, inhibition_detector_id=None, backoff_s=3.0),
        ],
        config=make_pipeline_config(),
    )
    dets = pipe.run_offline()
    sw_dets = get_detections(dets)
    sw = [e for e in gt if e.metadata.get('type') == 'SW']
    anns = [Annotation(timestamp=e.timestamp, channel=e.channel_id, event_type='SW') for e in sw]
    r = validate(sw_dets, anns, time_tolerance=CFG['validation']['time_tolerance'])
    results.append({'snr': actual_snr, **r.metrics, 'n_detected': len(sw_dets)})
    print(f'SNR={actual_snr:.1f}: P={r.metrics["precision"]:.2f} '
          f'R={r.metrics["recall"]:.2f} F1={r.metrics["f1"]:.2f}')

In [ ]:
snrs = [r['snr'] for r in results]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(snrs, [r['precision'] for r in results], 'o-', label='Precision', lw=2)
axes[0].plot(snrs, [r['recall'] for r in results], 's-', label='Recall', lw=2)
axes[0].plot(snrs, [r['f1'] for r in results], '^-', label='F1', lw=2)
axes[0].set_xlabel('SNR')
axes[0].set_ylabel('Score')
axes[0].set_ylim(-0.05, 1.05)
axes[0].set_title('Detection Performance vs SNR')
axes[0].legend()
axes[0].grid(alpha=0.3)

x = np.arange(len(snrs))
w = 0.25
axes[1].bar(x - w, [r['true_positives'] for r in results], w, label='TP')
axes[1].bar(x, [r['false_positives'] for r in results], w, label='FP')
axes[1].bar(x + w, [r['false_negatives'] for r in results], w, label='FN')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'{s:.0f}' for s in snrs])
axes[1].set_xlabel('SNR')
axes[1].set_ylabel('Count')
axes[1].set_title('Detection Breakdown')
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## 6. Inspect wavelet output

In [ ]:
from dnb.modules.base import ProcessResult
from dnb.core.ring_buffer import RingBuffer
from dnb.core.types import DataChunk

path_short, gt_short, sig_short = td.short_segment()

config = make_pipeline_config()
wavelet_mod = make_wavelet()
wavelet_mod.configure(config)

ring = RingBuffer(n_channels=1, capacity=config.buffer_samples)
chunk_data = sig_short[:, :config.chunk_samples]
ring.write(chunk_data)

chunk = DataChunk(
    samples=chunk_data,
    timestamps=np.arange(config.chunk_samples) / config.sample_rate,
    channel_ids=np.array([0], dtype=np.int32),
    sample_rate=config.sample_rate,
)
result = ProcessResult(chunk=chunk, ring_buffer=ring)
result = wavelet_mod.process(result)

wav = result.wavelet
t_chunk = chunk.timestamps

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

axes[0].plot(t_chunk, chunk.samples[0], 'b-', lw=0.5)
axes[0].set_ylabel('Raw signal')
axes[0].set_title(f'Wavelet decomposition (chunk_duration={config.chunk_duration}s, settled={result.wavelet_settled})')

im = axes[1].pcolormesh(t_chunk, wav.frequencies, wav.amplitude[0], shading='auto', cmap='hot')
axes[1].set_ylabel('Frequency (Hz)')
axes[1].set_yscale('log')
plt.colorbar(im, ax=axes[1], label='Amplitude')

tw = CFG['target_wave']
so_mask = (wav.frequencies >= tw['freq_range'][0]) & (wav.frequencies <= tw['freq_range'][1])
so_phase = np.mean(wav.phase[0, so_mask, :], axis=0)
axes[2].plot(t_chunk, so_phase, 'g-', lw=0.5)
axes[2].axhline(tw['target_phase'], color='red', ls='--', alpha=0.5,
                label=f'target phase ({tw["target_phase"]:.2f} rad)')
axes[2].set_ylabel('SO Phase (rad)')
axes[2].set_xlabel('Time (s)')
axes[2].legend()

plt.tight_layout()
plt.show()

---
## 7. Parameter exploration

In [ ]:
path_pe, gt_pe = td.slow_waves(snr=syn['snr'], seed=99)
sw_pe = [e for e in gt_pe if e.metadata.get('type') == 'SW']
anns_pe = [Annotation(timestamp=e.timestamp, channel=e.channel_id, event_type='SW') for e in sw_pe]

tolerances = [0.05, 0.1, 0.2, 0.3, 0.5, 0.8]
tol_results = []

for tol in tolerances:
    pipe = Pipeline(
        source=FileSource(str(path_pe)),
        modules=[
            make_wavelet(),
            make_detector(target_phase=pi, phase_tolerance=tol),
            make_trigger(n_pulses=0, inhibition_detector_id=None, backoff_s=3.0),
        ],
        config=make_pipeline_config(),
    )
    dets = pipe.run_offline()
    sw_dets = get_detections(dets)
    r = validate(sw_dets, anns_pe, time_tolerance=CFG['validation']['time_tolerance'])
    tol_results.append({'tolerance': tol, **r.metrics, 'n_detected': len(sw_dets)})
    print(f'tol={tol:.2f}: {len(sw_dets)} detections, '
          f'P={r.metrics["precision"]:.2f} R={r.metrics["recall"]:.2f}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(tolerances, [r['precision'] for r in tol_results], 'o-', label='Precision')
ax.plot(tolerances, [r['recall'] for r in tol_results], 's-', label='Recall')
ax.plot(tolerances, [r['f1'] for r in tol_results], '^-', label='F1')
ax.set_xlabel('Phase tolerance (rad)')
ax.set_ylabel('Score')
ax.set_title('Effect of phase tolerance on detection')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()